# Module 05 — Neural Network Foundations
## 05-09: Optimisers

**Objective:** Derive and implement SGD, Momentum SGD, AdaGrad, RMSProp, Adam, and
AdamW from scratch; understand *why* each improvement was invented; and compare
convergence speed and final accuracy on FashionMNIST.

**Prerequisites:** 05-06 (Backpropagation — gradient computation);
05-07 (PyTorch Autograd — `.backward()` and `param.grad`);
05-08 (Weight Initialisation — variance control before training starts).


## Part 0 — Setup & Prerequisites

Deep learning optimisers all solve the same problem — minimise a loss
$\mathcal{L}(\boldsymbol{\theta})$ by repeatedly moving parameters in the
direction of the negative gradient — but they differ in *how* they scale and
accumulate past gradient information.

The broad progression:

1. **Vanilla SGD** — noisy gradient step (baseline, easy to analyse).
2. **Momentum SGD** — exponentially-weighted running gradient (dampens oscillations).
3. **AdaGrad** — per-parameter learning rates scaled by *sum* of past squared gradients.
4. **RMSProp** — fixes AdaGrad's dying-LR problem with an *exponential* average.
5. **Adam** — momentum **+** RMSProp **+** bias correction (most widely used).
6. **AdamW** — Adam with *decoupled* weight decay (modern standard for Transformers).

> **Concept ownership:** This notebook owns all six optimiser algorithms.
> Weight decay / regularisation is introduced in **05-05**;
> learning-rate schedules are covered in **Module 16**.


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import time
import warnings
warnings.filterwarnings("ignore")

from collections.abc import Callable, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import random
print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy:   {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA:    {torch.version.cuda}")
    print(f"GPU:     {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Reproducibility & Device ──────────────────────────────────────────────────

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# FashionMNIST
FASHION_MEAN  = (0.2860,)
FASHION_STD   = (0.3530,)
NUM_CLASSES   = 10
INPUT_DIM     = 784       # 28 × 28
FASHION_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

# Architecture
HIDDEN_DIM = 256
ARCH       = [INPUT_DIM, HIDDEN_DIM, HIDDEN_DIM, NUM_CLASSES]

# Training
BATCH_SIZE    = 256
NUM_EPOCHS    = 10
LEARNING_RATE = 1e-3      # default; individual opts use this unless specified
WEIGHT_DECAY  = 1e-4      # used by AdamW only

# Toy optimisation experiment
N_TOY_STEPS = 200


### Data — FashionMNIST

FashionMNIST is a 10-class clothing image dataset with official train (60 000) and
test (10 000) splits.  We take a **90/10 split of the official training set** for
train/val.  Images are flattened inside the model's `forward` method.


In [ ]:
# ── FashionMNIST dataset ───────────────────────────────────────────────────────
transform_f = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(FASHION_MEAN, FASHION_STD),
])

full_train_f = torchvision.datasets.FashionMNIST(
    root="data/", train=True,  download=True, transform=transform_f
)
test_set_f = torchvision.datasets.FashionMNIST(
    root="data/", train=False, download=True, transform=transform_f
)

generator_f = torch.Generator().manual_seed(SEED)
train_size_f = int(0.9 * len(full_train_f))
val_size_f   = len(full_train_f) - train_size_f
train_set_f, val_set_f = torch.utils.data.random_split(
    full_train_f, [train_size_f, val_size_f], generator=generator_f
)

train_loader = DataLoader(train_set_f, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_set_f,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_set_f,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())

print(f"Train: {len(train_set_f):,}  Val: {len(val_set_f):,}  Test: {len(test_set_f):,}")
print(f"Batches per epoch: {len(train_loader)}")

# ── Quick EDA: class distribution ─────────────────────────────────────────────
all_labels = [label for _, label in full_train_f]
counts = np.bincount(all_labels)
print("\nClass counts (train+val):")
for idx, (name, cnt) in enumerate(zip(FASHION_CLASSES, counts)):
    print(f"  {idx:2d}  {name:<15s}  {cnt:5,}  ({cnt / len(all_labels) * 100:.1f}%)")


---
## Part 1 — From Scratch: Six Optimiser Families

We implement each optimiser as a simple Python class that operates on **plain NumPy
arrays**.  This separates the *update equation* from PyTorch's autograd machinery,
making each formula visible and testable.

### Common Interface

```python
opt.initialise(params)          # allocate state buffers
opt.step(params, grads)         # apply one parameter update (in-place)
opt.reset_state()               # zero all accumulated state
```

All from-scratch classes share this interface so we can run head-to-head comparisons
on a simple toy problem before connecting to FashionMNIST in Part 3.


### 1.1 SGD and Momentum SGD

**Vanilla SGD** takes a step proportional to the raw negative gradient:

$$\boldsymbol{\theta}_{t} = \boldsymbol{\theta}_{t-1} - \alpha \, \mathbf{g}_t$$

On *ill-conditioned* landscapes (highly anisotropic curvature) vanilla SGD zig-zags:
it takes large steps in directions with high curvature and small steps where the
gradient is small.

**Momentum** (Polyak, 1964) accumulates a velocity vector that dampens oscillations:

$$\mathbf{v}_{t} = \mu \, \mathbf{v}_{t-1} + \mathbf{g}_t, \qquad
\boldsymbol{\theta}_{t} = \boldsymbol{\theta}_{t-1} - \alpha \, \mathbf{v}_t$$

Intuitively, $\mu = 0.9$ means the velocity is a weighted average of the last
$\approx 1/(1-\mu) = 10$ gradients.


In [ ]:
# ── SGD from scratch (vanilla + Polyak momentum) ─────────────────────────────

class SGDScratch:
    '''Stochastic Gradient Descent with optional Polyak momentum.

    Update rule (momentum form, μ=0 reduces to vanilla SGD):
        v_t = μ * v_{t-1} + g_t
        θ_t = θ_{t-1} - α * v_t

    Attributes:
        lr:         Learning rate α.
        momentum:   Momentum coefficient μ ∈ [0, 1).
        velocities: Per-parameter velocity buffers.
    '''

    def __init__(self, lr: float, momentum: float = 0.0) -> None:
        '''Initialise SGD.

        Args:
            lr:       Step size (learning rate).
            momentum: Momentum coefficient (0 = vanilla SGD).
        '''
        self.lr         = lr
        self.momentum   = momentum
        self.velocities: list[np.ndarray] = []

    def initialise(self, params: list[np.ndarray]) -> None:
        '''Allocate zero velocity buffers matching each parameter shape.

        Args:
            params: List of parameter arrays.
        '''
        self.velocities = [np.zeros_like(p) for p in params]

    def step(
        self,
        params: list[np.ndarray],
        grads:  list[np.ndarray],
    ) -> None:
        '''Apply one SGD update (in-place).

        Args:
            params: Parameter arrays to update.
            grads:  Corresponding gradient arrays.
        '''
        for i, (p, g) in enumerate(zip(params, grads)):
            if self.momentum > 0.0:
                self.velocities[i] = self.momentum * self.velocities[i] + g
                p -= self.lr * self.velocities[i]
            else:
                p -= self.lr * g

    def reset_state(self) -> None:
        '''Zero all velocity buffers.'''
        self.velocities = [np.zeros_like(v) for v in self.velocities]


# ── Verify on 1-D convex: f(x) = x^2 + 3x + 2, minimum at x = -1.5 ─────────
x_v = np.array([3.0])
sgd_v = SGDScratch(lr=0.1, momentum=0.0)
sgd_v.initialise([x_v])
for _ in range(30):
    g_v = 2.0 * x_v + 3.0
    sgd_v.step([x_v], [g_v])
print(f"SGD (vanilla): 30 steps from x=3.0 → x={x_v[0]:.6f}  (target: -1.5)")

# ── Momentum dampens zig-zagging on ill-conditioned landscapes ─────────────────
# f(w1, w2) = w1^2 + 20*w2^2  (condition number = 20)
# ∇f = [2*w1, 40*w2]

def quad_grad(w: np.ndarray) -> np.ndarray:
    '''Gradient of the anisotropic quadratic f = w1^2 + 20*w2^2.

    Args:
        w: Parameter vector of shape (2,).

    Returns:
        Gradient vector [2*w1, 40*w2].
    '''
    return np.array([2.0 * w[0], 40.0 * w[1]])


w_start = np.array([-3.0, 2.0])
sgd_traces: dict[str, list[np.ndarray]] = {}

for mu, label in [(0.0, "SGD (vanilla)"), (0.9, "SGD (mu=0.9)")]:
    w = w_start.copy()
    opt_s = SGDScratch(lr=0.02, momentum=mu)
    opt_s.initialise([w])
    trace: list[np.ndarray] = [w.copy()]
    for _ in range(N_TOY_STEPS):
        g = quad_grad(w)
        opt_s.step([w], [g])
        trace.append(w.copy())
    sgd_traces[label] = trace
    print(f"  {label:<20s}  final dist={np.linalg.norm(w):.6f}")

# Contour plot: vanilla vs momentum trajectories
w1_g = np.linspace(-3.5, 0.5, 200)
w2_g = np.linspace(-2.5, 2.5, 200)
W1, W2  = np.meshgrid(w1_g, w2_g)
F_quad  = W1 ** 2 + 20 * W2 ** 2

fig_sgd, ax_sgd = plt.subplots(figsize=(9, 6))
ax_sgd.contourf(W1, W2, F_quad, levels=25, cmap="Blues", alpha=0.55)
ax_sgd.contour( W1, W2, F_quad, levels=15, colors="white", linewidths=0.5, alpha=0.4)

traj_colors = {"SGD (vanilla)": "#d62728", "SGD (mu=0.9)": "#2ca02c"}
for label, trace in sgd_traces.items():
    arr = np.array(trace)
    ax_sgd.plot(arr[:, 0], arr[:, 1], "o-", lw=1.5, ms=2,
                color=traj_colors[label], label=label)

ax_sgd.plot(0, 0, "*", ms=14, color="gold", zorder=5, label="Minimum (0, 0)")
ax_sgd.set_xlabel("w₁", fontsize=11)
ax_sgd.set_ylabel("w₂", fontsize=11)
ax_sgd.set_title("SGD Vanilla vs Momentum on Anisotropic Quadratic (cond=20)",
                 fontsize=11, fontweight="bold")
ax_sgd.legend(fontsize=9)
plt.tight_layout()
plt.show()


### 1.2 AdaGrad and RMSProp — Adaptive Learning Rates

**AdaGrad** (Duchi et al., 2011) gives each parameter its own learning rate,
inversely scaled by the root of the *sum* of all past squared gradients:

$$G_t = G_{t-1} + \mathbf{g}_t \odot \mathbf{g}_t, \qquad
\boldsymbol{\theta}_t = \boldsymbol{\theta}_{t-1}
- \frac{\alpha}{\sqrt{G_t + \varepsilon}} \odot \mathbf{g}_t$$

**Problem:** $G_t$ grows monotonically → learning rate decays to zero even for
parameters that are still far from convergence.

**RMSProp** (Hinton, 2012) replaces the cumulative sum with an *exponential moving
average*, so $G_t$ stops growing when gradients stabilise:

$$v_t = \rho \, v_{t-1} + (1-\rho) \, \mathbf{g}_t^2, \qquad
\boldsymbol{\theta}_t = \boldsymbol{\theta}_{t-1}
- \frac{\alpha}{\sqrt{v_t + \varepsilon}} \odot \mathbf{g}_t$$


In [ ]:
# ── AdaGrad from scratch ──────────────────────────────────────────────────────

class AdaGradScratch:
    '''AdaGrad: adapts learning rates via cumulative squared-gradient sum.

    G grows monotonically, so the effective learning rate α/sqrt(G) shrinks
    throughout training.  Excellent for sparse features; problematic for dense
    ones where the learning rate can die before convergence.

    Attributes:
        lr:  Global learning rate.
        eps: Stability constant added inside the square root.
        G:   Per-parameter accumulated squared gradients.
    '''

    def __init__(self, lr: float, eps: float = 1e-8) -> None:
        '''Initialise AdaGrad.

        Args:
            lr:  Learning rate.
            eps: Numerical stability constant.
        '''
        self.lr  = lr
        self.eps = eps
        self.G: list[np.ndarray] = []

    def initialise(self, params: list[np.ndarray]) -> None:
        '''Allocate zero G accumulators.

        Args:
            params: List of parameter arrays.
        '''
        self.G = [np.zeros_like(p) for p in params]

    def step(
        self,
        params: list[np.ndarray],
        grads:  list[np.ndarray],
    ) -> None:
        '''Apply one AdaGrad update (in-place).

        Args:
            params: Parameter arrays (updated in-place).
            grads:  Gradient arrays.
        '''
        for i, (p, g) in enumerate(zip(params, grads)):
            self.G[i] += g * g
            p -= self.lr / (np.sqrt(self.G[i]) + self.eps) * g

    def reset_state(self) -> None:
        '''Zero G accumulators.'''
        self.G = [np.zeros_like(g_acc) for g_acc in self.G]


# ── RMSProp from scratch ───────────────────────────────────────────────────────

class RMSPropScratch:
    '''RMSProp: adaptive LR with exponentially-decaying squared-gradient average.

    Fixes AdaGrad's monotonically-decaying LR by replacing the cumulative sum
    with an exponential moving average (EMA) of squared gradients.

    Attributes:
        lr:    Learning rate.
        alpha: EMA decay coefficient ρ (typically 0.99).
        eps:   Stability constant.
        v:     Per-parameter EMA of squared gradients.
    '''

    def __init__(self, lr: float, alpha: float = 0.99, eps: float = 1e-8) -> None:
        '''Initialise RMSProp.

        Args:
            lr:    Learning rate.
            alpha: Decay coefficient for the squared-gradient EMA.
            eps:   Numerical stability constant.
        '''
        self.lr    = lr
        self.alpha = alpha
        self.eps   = eps
        self.v: list[np.ndarray] = []

    def initialise(self, params: list[np.ndarray]) -> None:
        '''Allocate zero EMA buffers.

        Args:
            params: List of parameter arrays.
        '''
        self.v = [np.zeros_like(p) for p in params]

    def step(
        self,
        params: list[np.ndarray],
        grads:  list[np.ndarray],
    ) -> None:
        '''Apply one RMSProp update (in-place).

        Args:
            params: Parameter arrays (updated in-place).
            grads:  Gradient arrays.
        '''
        for i, (p, g) in enumerate(zip(params, grads)):
            self.v[i] = self.alpha * self.v[i] + (1.0 - self.alpha) * g * g
            p -= self.lr / (np.sqrt(self.v[i]) + self.eps) * g

    def reset_state(self) -> None:
        '''Zero EMA buffers.'''
        self.v = [np.zeros_like(vi) for vi in self.v]


# ── Compare AdaGrad vs RMSProp on quadratic: show G vs v over time ────────────
w_ag = w_start.copy()
w_rm = w_start.copy()

opt_ag = AdaGradScratch(lr=0.15)
opt_rm = RMSPropScratch(lr=0.05, alpha=0.9)
opt_ag.initialise([w_ag])
opt_rm.initialise([w_rm])

g_norms_ag: list[float] = []
g_norms_rm: list[float] = []
eff_lr_ag:  list[float] = []
eff_lr_rm:  list[float] = []

for step_i in range(N_TOY_STEPS):
    g_ag = quad_grad(w_ag)
    g_rm = quad_grad(w_rm)
    opt_ag.step([w_ag], [g_ag])
    opt_rm.step([w_rm], [g_rm])
    g_norms_ag.append(float(np.linalg.norm(g_ag)))
    g_norms_rm.append(float(np.linalg.norm(g_rm)))
    eff_lr_ag.append(float(opt_ag.lr / (np.sqrt(opt_ag.G[0][0]) + opt_ag.eps)))
    eff_lr_rm.append(float(opt_rm.lr / (np.sqrt(opt_rm.v[0][0]) + opt_rm.eps)))

print(f"AdaGrad  final dist: {np.linalg.norm(w_ag):.6f}")
print(f"RMSProp  final dist: {np.linalg.norm(w_rm):.6f}")

fig_ada, axes_ada = plt.subplots(1, 2, figsize=(13, 4))
steps_ax = list(range(N_TOY_STEPS))
axes_ada[0].plot(steps_ax, eff_lr_ag, color="#d62728", lw=1.5, label="AdaGrad eff. LR")
axes_ada[0].plot(steps_ax, eff_lr_rm, color="#1f77b4", lw=1.5, label="RMSProp eff. LR")
axes_ada[0].set_xlabel("Step", fontsize=11)
axes_ada[0].set_ylabel("Effective learning rate", fontsize=11)
axes_ada[0].set_title("AdaGrad LR Decays to Zero; RMSProp Stabilises",
                       fontsize=11, fontweight="bold")
axes_ada[0].legend(fontsize=9)
axes_ada[0].grid(alpha=0.3)
axes_ada[1].semilogy(steps_ax, g_norms_ag, color="#d62728", lw=1.5, label="AdaGrad")
axes_ada[1].semilogy(steps_ax, g_norms_rm, color="#1f77b4", lw=1.5, label="RMSProp")
axes_ada[1].set_xlabel("Step", fontsize=11)
axes_ada[1].set_ylabel("||grad|| (log)", fontsize=11)
axes_ada[1].set_title("Gradient Norm vs Step", fontsize=11, fontweight="bold")
axes_ada[1].legend(fontsize=9)
axes_ada[1].grid(alpha=0.3)
plt.suptitle("AdaGrad vs RMSProp: Effective Learning Rate Comparison",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


### 1.3 Adam and AdamW — Momentum + Adaptive LR + Bias Correction

**Adam** (Kingma & Ba, 2015) combines:
- A first-moment estimate (like momentum): $m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t$
- A second-moment estimate (like RMSProp): $v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$
- **Bias correction** to account for the zero initialisation of $m$ and $v$:

$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$

$$\boldsymbol{\theta}_t = \boldsymbol{\theta}_{t-1}
- \frac{\alpha}{\sqrt{\hat{v}_t}+\varepsilon} \hat{m}_t$$

**AdamW** (Loshchilov & Hutter, 2019) fixes a subtle bug: in Adam, weight decay
(L2 regularisation) is folded into the gradient and then *scaled down* by
$1/\sqrt{\hat{v}_t}$, reducing its effect.  AdamW applies weight decay *directly*
to the parameters, independent of the adaptive scaling:

$$\boldsymbol{\theta}_t = (1 - \alpha \lambda) \boldsymbol{\theta}_{t-1}
- \frac{\alpha}{\sqrt{\hat{v}_t}+\varepsilon} \hat{m}_t$$


In [ ]:
# ── Adam from scratch ─────────────────────────────────────────────────────────

class AdamScratch:
    '''Adam optimiser: momentum + adaptive LR + bias correction.

    Combines first-moment (momentum) and second-moment (RMSProp) estimates
    with bias correction to account for zero initialisation.

    Attributes:
        lr:    Learning rate α.
        beta1: Exponential decay for first moment (typically 0.9).
        beta2: Exponential decay for second moment (typically 0.999).
        eps:   Numerical stability constant.
        m:     First-moment estimates per parameter.
        v:     Second-moment estimates per parameter.
        t:     Step counter for bias correction.
    '''

    def __init__(
        self,
        lr:    float = 1e-3,
        beta1: float = 0.9,
        beta2: float = 0.999,
        eps:   float = 1e-8,
    ) -> None:
        '''Initialise Adam.

        Args:
            lr:    Learning rate.
            beta1: First-moment decay (momentum).
            beta2: Second-moment decay (adaptive scaling).
            eps:   Stability constant.
        '''
        self.lr    = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps   = eps
        self.m:    list[np.ndarray] = []
        self.v:    list[np.ndarray] = []
        self.t:    int              = 0

    def initialise(self, params: list[np.ndarray]) -> None:
        '''Allocate zero moment buffers.

        Args:
            params: List of parameter arrays.
        '''
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def step(
        self,
        params: list[np.ndarray],
        grads:  list[np.ndarray],
    ) -> None:
        '''Apply one Adam update (in-place).

        Args:
            params: Parameter arrays (updated in-place).
            grads:  Gradient arrays.
        '''
        self.t += 1
        bc1 = 1.0 - self.beta1 ** self.t   # bias correction denominator for m
        bc2 = 1.0 - self.beta2 ** self.t   # bias correction denominator for v
        for i, (p, g) in enumerate(zip(params, grads)):
            self.m[i] = self.beta1 * self.m[i] + (1.0 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1.0 - self.beta2) * g * g
            m_hat = self.m[i] / bc1
            v_hat = self.v[i] / bc2
            p -= self.lr / (np.sqrt(v_hat) + self.eps) * m_hat

    def reset_state(self) -> None:
        '''Zero all moment buffers and reset step counter.'''
        self.m = [np.zeros_like(mi) for mi in self.m]
        self.v = [np.zeros_like(vi) for vi in self.v]
        self.t = 0


# ── AdamW from scratch ────────────────────────────────────────────────────────

class AdamWScratch:
    '''AdamW: Adam with decoupled weight decay.

    Applies weight decay directly to parameters (not via gradients), so
    the adaptive scaling does not reduce its effective strength.

    Attributes:
        lr:           Learning rate α.
        beta1:        First-moment decay.
        beta2:        Second-moment decay.
        eps:          Numerical stability constant.
        weight_decay: L2 regularisation coefficient λ.
        m, v, t:      Moment buffers and step counter (same as Adam).
    '''

    def __init__(
        self,
        lr:           float = 1e-3,
        beta1:        float = 0.9,
        beta2:        float = 0.999,
        eps:          float = 1e-8,
        weight_decay: float = 1e-2,
    ) -> None:
        '''Initialise AdamW.

        Args:
            lr:           Learning rate.
            beta1:        First-moment decay.
            beta2:        Second-moment decay.
            eps:          Stability constant.
            weight_decay: Decoupled weight decay coefficient λ.
        '''
        self.lr           = lr
        self.beta1        = beta1
        self.beta2        = beta2
        self.eps          = eps
        self.weight_decay = weight_decay
        self.m: list[np.ndarray] = []
        self.v: list[np.ndarray] = []
        self.t: int              = 0

    def initialise(self, params: list[np.ndarray]) -> None:
        '''Allocate zero moment buffers.

        Args:
            params: List of parameter arrays.
        '''
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def step(
        self,
        params: list[np.ndarray],
        grads:  list[np.ndarray],
    ) -> None:
        '''Apply one AdamW update (in-place).

        Args:
            params: Parameter arrays (updated in-place).
            grads:  Gradient arrays.
        '''
        self.t += 1
        bc1 = 1.0 - self.beta1 ** self.t
        bc2 = 1.0 - self.beta2 ** self.t
        for i, (p, g) in enumerate(zip(params, grads)):
            p   *= (1.0 - self.lr * self.weight_decay)    # decoupled decay
            self.m[i] = self.beta1 * self.m[i] + (1.0 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1.0 - self.beta2) * g * g
            m_hat = self.m[i] / bc1
            v_hat = self.v[i] / bc2
            p -= self.lr / (np.sqrt(v_hat) + self.eps) * m_hat

    def reset_state(self) -> None:
        '''Zero all moment buffers and reset step counter.'''
        self.m = [np.zeros_like(mi) for mi in self.m]
        self.v = [np.zeros_like(vi) for vi in self.v]
        self.t = 0


# ── Bias correction demo: why initialising m, v to 0 introduces bias ─────────
print("Bias correction demo — first 5 Adam steps on g=1 (constant gradient):")
print(f"  {'Step':>4s}  {'m_t':>8s}  {'v_t':>8s}  {'m_hat':>8s}  {'v_hat':>8s}")
print("  " + "-" * 46)

beta1, beta2 = 0.9, 0.999
m_bc, v_bc   = 0.0, 0.0
for t_bc in range(1, 6):
    m_bc = beta1 * m_bc + (1 - beta1) * 1.0
    v_bc = beta2 * v_bc + (1 - beta2) * 1.0 ** 2
    m_hat_bc = m_bc / (1 - beta1 ** t_bc)
    v_hat_bc = v_bc / (1 - beta2 ** t_bc)
    print(f"  {t_bc:>4d}  {m_bc:>8.5f}  {v_bc:>8.5f}  {m_hat_bc:>8.5f}  {v_hat_bc:>8.5f}")
print("  Without correction, m and v are biased low in early steps.")
print("  m_hat and v_hat scale them up to the correct long-run values.")


---
## Part 2 — Toy Convergence Comparison

With all six from-scratch implementations in hand, we run them on the same
**2-D anisotropic quadratic**:

$$f(w_1, w_2) = w_1^2 + 20 w_2^2, \qquad \nabla f = [2w_1,\; 40w_2]$$

Starting point: $(-3, 2)$.  Minimum: $(0, 0)$.

The high condition number (20) exaggerates the difference between plain SGD
(zig-zags) and adaptive methods (take nearly-optimal steps per dimension).


In [ ]:
# ── All 6 optimisers on the anisotropic quadratic ────────────────────────────
scratch_opts: dict[str, object] = {
    "SGD (vanilla)":    SGDScratch(lr=0.02,  momentum=0.0),
    "SGD (mu=0.9)":     SGDScratch(lr=0.02,  momentum=0.9),
    "AdaGrad":          AdaGradScratch(lr=0.3),
    "RMSProp":          RMSPropScratch(lr=0.05, alpha=0.9),
    "Adam":             AdamScratch(lr=0.08),
    "AdamW":            AdamWScratch(lr=0.08, weight_decay=0.01),
}

all_traces:     dict[str, list[np.ndarray]] = {}
conv_distances: dict[str, list[float]]      = {}

for name, opt in scratch_opts.items():
    w = w_start.copy()
    opt.initialise([w])
    trace_w: list[np.ndarray] = [w.copy()]
    dists:   list[float]      = [float(np.linalg.norm(w))]
    for _ in range(N_TOY_STEPS):
        g = quad_grad(w)
        opt.step([w], [g])
        trace_w.append(w.copy())
        dists.append(float(np.linalg.norm(w)))
    all_traces[name]     = trace_w
    conv_distances[name] = dists
    print(f"  {name:<20s}  final dist={dists[-1]:.6f}")

# ── 2-D trajectory plot ────────────────────────────────────────────────────────
fig_all, ax_all = plt.subplots(figsize=(10, 7))
ax_all.contourf(W1, W2, F_quad, levels=30, cmap="Greys", alpha=0.5)
ax_all.contour( W1, W2, F_quad, levels=15, colors="grey", linewidths=0.4, alpha=0.5)

traj_cols = ["#d62728", "#ff7f0e", "#2ca02c", "#1f77b4", "#9467bd", "#8c564b"]
for (name, trace), col in zip(all_traces.items(), traj_cols):
    arr = np.array(trace)
    ax_all.plot(arr[:, 0], arr[:, 1], "-", lw=1.5, color=col, label=name, alpha=0.85)
    ax_all.plot(arr[0, 0],  arr[0, 1],  "o", ms=6, color=col)
    ax_all.plot(arr[-1, 0], arr[-1, 1], "s", ms=5, color=col)

ax_all.plot(0, 0, "*", ms=16, color="gold", zorder=6, label="Minimum (0, 0)")
ax_all.set_xlabel("w₁", fontsize=11)
ax_all.set_ylabel("w₂", fontsize=11)
ax_all.set_title(f"All 6 Optimisers on f = w₁² + 20w₂²  ({N_TOY_STEPS} steps)",
                 fontsize=11, fontweight="bold")
ax_all.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

# ── Convergence curve: distance to minimum vs step ────────────────────────────
fig_dist, ax_dist = plt.subplots(figsize=(10, 4))
for (name, dists), col in zip(conv_distances.items(), traj_cols):
    ax_dist.semilogy(dists, lw=1.8, color=col, label=name)
ax_dist.set_xlabel("Step", fontsize=11)
ax_dist.set_ylabel("Distance to minimum (log)", fontsize=11)
ax_dist.set_title("Convergence Speed: Distance to Minimum vs Step",
                  fontsize=11, fontweight="bold")
ax_dist.legend(fontsize=9)
ax_dist.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ── Summary table: steps to reach 5 distance thresholds ─────────────────────
thresholds = [2.0, 1.0, 0.5, 0.1, 0.01]
conv_rows: list[dict] = []
for name, dists in conv_distances.items():
    row: dict = {"Optimiser": name}
    for thr in thresholds:
        steps_to = next((s for s, d in enumerate(dists) if d < thr), None)
        row[f"<{thr}"] = steps_to if steps_to is not None else ">"
    conv_rows.append(row)
conv_df = pd.DataFrame(conv_rows)
print("\nSteps to reach distance threshold (\">\": not reached in 200 steps):")
print(conv_df.to_string(index=False))


---
## Part 3 — Training on FashionMNIST

We now connect the optimisers to a real dataset.  Each optimiser is used to train
the **same MLP** (784 → 256 → 256 → 10, ReLU) from the **same initialisation** for
10 epochs on FashionMNIST.  The only difference between runs is the optimiser.

We implement each as a proper `torch.optim.Optimizer` subclass so they integrate
seamlessly with PyTorch's autograd and parameter groups.


In [ ]:
# ── FashionMNIST MLP ──────────────────────────────────────────────────────────

class FashionMLP(nn.Module):
    '''Three-hidden-layer MLP for FashionMNIST classification.

    Attributes:
        net: Sequential stack of Linear → ReLU layers plus output Linear.
    '''

    def __init__(self, layer_sizes: list[int]) -> None:
        '''Build the MLP.

        Args:
            layer_sizes: List [n_in, h1, ..., n_out] of layer widths.
        '''
        super().__init__()
        layers: list[nn.Module] = []
        for fan_in, fan_out in zip(layer_sizes[:-1], layer_sizes[1:]):
            layers.append(nn.Linear(fan_in, fan_out))
            if fan_out != layer_sizes[-1]:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Forward pass: flatten then apply MLP.

        Args:
            x: Input of shape (N, 1, 28, 28) or (N, 784).

        Returns:
            Class logits of shape (N, num_classes).
        '''
        return self.net(x.view(x.size(0), -1))


def train_one_epoch(
    model:      nn.Module,
    dataloader: DataLoader,
    optimizer:  optim.Optimizer,
    criterion:  nn.Module,
    device:     torch.device,
) -> tuple[float, float]:
    '''Train for one epoch and return (avg_loss, accuracy).

    Args:
        model:      Neural network to train.
        dataloader: Training DataLoader.
        optimizer:  Optimizer instance.
        criterion:  Loss function.
        device:     Compute device.

    Returns:
        Tuple of (average_loss, accuracy).
    '''
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for X_b, y_b in dataloader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        out  = model(X_b)
        loss = criterion(out, y_b)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_b.size(0)
        correct      += out.argmax(1).eq(y_b).sum().item()
        total        += X_b.size(0)
    return running_loss / total, correct / total


def evaluate(
    model:      nn.Module,
    dataloader: DataLoader,
    criterion:  nn.Module,
    device:     torch.device,
) -> tuple[float, float]:
    '''Evaluate model and return (avg_loss, accuracy).

    Args:
        model:      Neural network to evaluate.
        dataloader: DataLoader for evaluation split.
        criterion:  Loss function.
        device:     Compute device.

    Returns:
        Tuple of (average_loss, accuracy).
    '''
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_b, y_b in dataloader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            out  = model(X_b)
            loss = criterion(out, y_b)
            running_loss += loss.item() * X_b.size(0)
            correct      += out.argmax(1).eq(y_b).sum().item()
            total        += X_b.size(0)
    return running_loss / total, correct / total


n_params = sum(p.numel() for p in FashionMLP(ARCH).parameters())
print(f"Architecture: {ARCH}")
print(f"Parameters:   {n_params:,}")


In [ ]:
# ── PyTorch Optimizer subclasses ─────────────────────────────────────────────

class SGDFromScratch(optim.Optimizer):
    '''SGD with optional Polyak momentum (PyTorch Optimizer interface).'''

    def __init__(
        self,
        params: Iterable,
        lr:       float = 1e-2,
        momentum: float = 0.0,
    ) -> None:
        '''Initialise SGD.

        Args:
            params:   Iterable of parameters or parameter groups.
            lr:       Learning rate.
            momentum: Momentum coefficient (0 = vanilla SGD).
        '''
        defaults = dict(lr=lr, momentum=momentum)
        super().__init__(params, defaults)

    def step(self, closure: Callable | None = None) -> float | None:
        '''Apply one SGD parameter update.

        Args:
            closure: Optional closure that re-evaluates the model.

        Returns:
            Loss value if closure is provided, else None.
        '''
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            lr_g = group["lr"]
            mu_g = group["momentum"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g_p = p.grad.data
                if mu_g > 0.0:
                    if "v" not in self.state[p]:
                        self.state[p]["v"] = torch.zeros_like(p.data)
                    v_p = self.state[p]["v"]
                    v_p.mul_(mu_g).add_(g_p)
                    p.data.add_(-lr_g * v_p)
                else:
                    p.data.add_(-lr_g * g_p)
        return loss


class AdaGradFromScratch(optim.Optimizer):
    '''AdaGrad (PyTorch Optimizer interface).'''

    def __init__(self, params: Iterable, lr: float = 1e-2, eps: float = 1e-8) -> None:
        '''Initialise AdaGrad.

        Args:
            params: Iterable of parameters or parameter groups.
            lr:     Learning rate.
            eps:    Numerical stability constant.
        '''
        defaults = dict(lr=lr, eps=eps)
        super().__init__(params, defaults)

    def step(self, closure: Callable | None = None) -> float | None:
        '''Apply one AdaGrad parameter update.

        Args:
            closure: Optional closure.

        Returns:
            Loss if closure provided.
        '''
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            lr_g  = group["lr"]
            eps_g = group["eps"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g_p = p.grad.data
                if "G" not in self.state[p]:
                    self.state[p]["G"] = torch.zeros_like(p.data)
                G_p = self.state[p]["G"]
                G_p.addcmul_(g_p, g_p)
                p.data.addcdiv_(-lr_g * g_p, G_p.sqrt().add_(eps_g))
        return loss


class RMSPropFromScratch(optim.Optimizer):
    '''RMSProp (PyTorch Optimizer interface).'''

    def __init__(
        self,
        params: Iterable,
        lr:    float = 1e-2,
        alpha: float = 0.99,
        eps:   float = 1e-8,
    ) -> None:
        '''Initialise RMSProp.

        Args:
            params: Iterable of parameters or parameter groups.
            lr:     Learning rate.
            alpha:  EMA decay coefficient.
            eps:    Numerical stability constant.
        '''
        defaults = dict(lr=lr, alpha=alpha, eps=eps)
        super().__init__(params, defaults)

    def step(self, closure: Callable | None = None) -> float | None:
        '''Apply one RMSProp parameter update.

        Args:
            closure: Optional closure.

        Returns:
            Loss if closure provided.
        '''
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            lr_g    = group["lr"]
            alpha_g = group["alpha"]
            eps_g   = group["eps"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g_p = p.grad.data
                if "v" not in self.state[p]:
                    self.state[p]["v"] = torch.zeros_like(p.data)
                v_p = self.state[p]["v"]
                v_p.mul_(alpha_g).addcmul_((1.0 - alpha_g) * g_p, g_p)
                p.data.addcdiv_(-lr_g * g_p, v_p.sqrt().add_(eps_g))
        return loss


class AdamFromScratch(optim.Optimizer):
    '''Adam (PyTorch Optimizer interface).'''

    def __init__(
        self,
        params: Iterable,
        lr:    float = 1e-3,
        beta1: float = 0.9,
        beta2: float = 0.999,
        eps:   float = 1e-8,
    ) -> None:
        '''Initialise Adam.

        Args:
            params: Iterable of parameters or parameter groups.
            lr:     Learning rate.
            beta1:  First-moment decay.
            beta2:  Second-moment decay.
            eps:    Stability constant.
        '''
        defaults = dict(lr=lr, beta1=beta1, beta2=beta2, eps=eps)
        super().__init__(params, defaults)

    def step(self, closure: Callable | None = None) -> float | None:
        '''Apply one Adam parameter update.

        Args:
            closure: Optional closure.

        Returns:
            Loss if closure provided.
        '''
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            lr_g    = group["lr"]
            beta1_g = group["beta1"]
            beta2_g = group["beta2"]
            eps_g   = group["eps"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g_p = p.grad.data
                state = self.state[p]
                if "t" not in state:
                    state["t"] = 0
                    state["m"] = torch.zeros_like(p.data)
                    state["v"] = torch.zeros_like(p.data)
                state["t"] += 1
                t_p  = state["t"]
                m_p  = state["m"]
                v_p  = state["v"]
                m_p.mul_(beta1_g).add_((1.0 - beta1_g) * g_p)
                v_p.mul_(beta2_g).addcmul_((1.0 - beta2_g) * g_p, g_p)
                bc1  = 1.0 - beta1_g ** t_p
                bc2  = 1.0 - beta2_g ** t_p
                step_size = lr_g / bc1
                denom = (v_p.sqrt() / (bc2 ** 0.5)).add_(eps_g)
                p.data.addcdiv_(-step_size * m_p, denom)
        return loss


class AdamWFromScratch(optim.Optimizer):
    '''AdamW: Adam with decoupled weight decay (PyTorch Optimizer interface).'''

    def __init__(
        self,
        params: Iterable,
        lr:           float = 1e-3,
        beta1:        float = 0.9,
        beta2:        float = 0.999,
        eps:          float = 1e-8,
        weight_decay: float = 1e-2,
    ) -> None:
        '''Initialise AdamW.

        Args:
            params:       Iterable of parameters or parameter groups.
            lr:           Learning rate.
            beta1:        First-moment decay.
            beta2:        Second-moment decay.
            eps:          Stability constant.
            weight_decay: Decoupled weight decay coefficient.
        '''
        defaults = dict(lr=lr, beta1=beta1, beta2=beta2, eps=eps,
                        weight_decay=weight_decay)
        super().__init__(params, defaults)

    def step(self, closure: Callable | None = None) -> float | None:
        '''Apply one AdamW parameter update.

        Args:
            closure: Optional closure.

        Returns:
            Loss if closure provided.
        '''
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            lr_g    = group["lr"]
            beta1_g = group["beta1"]
            beta2_g = group["beta2"]
            eps_g   = group["eps"]
            wd_g    = group["weight_decay"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g_p = p.grad.data
                # Decoupled weight decay
                p.data.mul_(1.0 - lr_g * wd_g)
                state = self.state[p]
                if "t" not in state:
                    state["t"] = 0
                    state["m"] = torch.zeros_like(p.data)
                    state["v"] = torch.zeros_like(p.data)
                state["t"] += 1
                t_p  = state["t"]
                m_p  = state["m"]
                v_p  = state["v"]
                m_p.mul_(beta1_g).add_((1.0 - beta1_g) * g_p)
                v_p.mul_(beta2_g).addcmul_((1.0 - beta2_g) * g_p, g_p)
                bc1  = 1.0 - beta1_g ** t_p
                bc2  = 1.0 - beta2_g ** t_p
                step_size = lr_g / bc1
                denom = (v_p.sqrt() / (bc2 ** 0.5)).add_(eps_g)
                p.data.addcdiv_(-step_size * m_p, denom)
        return loss


print("All 5 PyTorch Optimizer subclasses defined.")
print("PyTorch built-ins used for comparison: optim.SGD, optim.Adam, optim.AdamW")


In [ ]:
# ── Train all optimisers on FashionMNIST ─────────────────────────────────────
# Shared model init state — we re-apply the same init to each model so
# the only variable is the optimiser.
torch.manual_seed(SEED)
ref_model     = FashionMLP(ARCH)
ref_state     = {k: v.clone() for k, v in ref_model.state_dict().items()}
criterion_cmp = nn.CrossEntropyLoss()

opt_configs: list[tuple[str, object]] = [
    ("SGD (vanilla)",   SGDFromScratch(ref_model.parameters(), lr=0.02)),
    ("SGD (mu=0.9)",    SGDFromScratch(ref_model.parameters(), lr=0.02, momentum=0.9)),
    ("AdaGrad",         AdaGradFromScratch(ref_model.parameters(), lr=0.01)),
    ("RMSProp",         RMSPropFromScratch(ref_model.parameters(), lr=0.001)),
    ("Adam",            AdamFromScratch(ref_model.parameters(), lr=LEARNING_RATE)),
    ("AdamW",           AdamWFromScratch(ref_model.parameters(), lr=LEARNING_RATE,
                                         weight_decay=WEIGHT_DECAY)),
    # PyTorch built-ins for verification
    ("PyTorch Adam",    optim.Adam(ref_model.parameters(), lr=LEARNING_RATE)),
    ("PyTorch AdamW",   optim.AdamW(ref_model.parameters(), lr=LEARNING_RATE,
                                    weight_decay=WEIGHT_DECAY)),
]

results_cmp: dict[str, dict] = {}

for opt_name, _ in opt_configs:
    torch.manual_seed(SEED)
    model_cmp = FashionMLP(ARCH).to(device)
    model_cmp.load_state_dict({k: v.clone() for k, v in ref_state.items()})

    # Rebuild the optimizer with the new model's parameters
    if opt_name == "SGD (vanilla)":
        opt_cmp = SGDFromScratch(model_cmp.parameters(), lr=0.02)
    elif opt_name == "SGD (mu=0.9)":
        opt_cmp = SGDFromScratch(model_cmp.parameters(), lr=0.02, momentum=0.9)
    elif opt_name == "AdaGrad":
        opt_cmp = AdaGradFromScratch(model_cmp.parameters(), lr=0.01)
    elif opt_name == "RMSProp":
        opt_cmp = RMSPropFromScratch(model_cmp.parameters(), lr=0.001)
    elif opt_name == "Adam":
        opt_cmp = AdamFromScratch(model_cmp.parameters(), lr=LEARNING_RATE)
    elif opt_name == "AdamW":
        opt_cmp = AdamWFromScratch(model_cmp.parameters(), lr=LEARNING_RATE,
                                    weight_decay=WEIGHT_DECAY)
    elif opt_name == "PyTorch Adam":
        opt_cmp = optim.Adam(model_cmp.parameters(), lr=LEARNING_RATE)
    else:
        opt_cmp = optim.AdamW(model_cmp.parameters(), lr=LEARNING_RATE,
                               weight_decay=WEIGHT_DECAY)

    tr_losses, vl_losses, tr_accs, vl_accs = [], [], [], []
    t0_cmp = time.time()
    for epoch in range(NUM_EPOCHS):
        tr_loss, tr_acc = train_one_epoch(
            model_cmp, train_loader, opt_cmp, criterion_cmp, device
        )
        vl_loss, vl_acc = evaluate(model_cmp, val_loader, criterion_cmp, device)
        tr_losses.append(tr_loss)
        vl_losses.append(vl_loss)
        tr_accs.append(tr_acc)
        vl_accs.append(vl_acc)
        if epoch == 0 or (epoch + 1) % 5 == 0:
            elapsed_cmp = time.time() - t0_cmp
            print(f"  [{opt_name}] Epoch {epoch+1}/{NUM_EPOCHS} | "
                  f"Train Loss: {tr_loss:.4f} | Val Loss: {vl_loss:.4f} | "
                  f"Val Acc: {vl_acc:.2%} | Time: {elapsed_cmp:.1f}s")

    test_loss, test_acc = evaluate(model_cmp, test_loader, criterion_cmp, device)
    results_cmp[opt_name] = {
        "train_losses": tr_losses,
        "val_losses":   vl_losses,
        "train_accs":   tr_accs,
        "val_accs":     vl_accs,
        "test_acc":     test_acc,
        "elapsed":      time.time() - t0_cmp,
    }
    print(f"  [{opt_name}] Final test acc: {test_acc:.2%}\n")


In [ ]:
# ── Training curve comparison ─────────────────────────────────────────────────
epochs_ax = list(range(1, NUM_EPOCHS + 1))
cmap_res  = plt.cm.tab10
n_opts    = len(results_cmp)
cols_res  = [cmap_res(i / n_opts) for i in range(n_opts)]

fig_tc, axes_tc = plt.subplots(1, 2, figsize=(14, 4))
for (opt_name, res), col in zip(results_cmp.items(), cols_res):
    axes_tc[0].plot(epochs_ax, res["val_losses"], lw=1.8, color=col, label=opt_name)
    axes_tc[1].plot(epochs_ax, res["val_accs"],   lw=1.8, color=col, label=opt_name)

for ax_tc in axes_tc:
    ax_tc.set_xlabel("Epoch", fontsize=11)
    ax_tc.legend(fontsize=7, loc="best")
    ax_tc.grid(alpha=0.3)

axes_tc[0].set_ylabel("Val Loss", fontsize=11)
axes_tc[0].set_title("Validation Loss by Optimiser", fontsize=11, fontweight="bold")
axes_tc[1].set_ylabel("Val Accuracy", fontsize=11)
axes_tc[1].set_title("Validation Accuracy by Optimiser", fontsize=11, fontweight="bold")
plt.suptitle("FashionMNIST Training: Optimiser Comparison (10 epochs)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
summary_rows = []
for opt_name, res in results_cmp.items():
    summary_rows.append({
        "Optimiser":      opt_name,
        "Epoch-1 Val Acc": f"{res['val_accs'][0]:.2%}",
        "Final Val Acc":   f"{res['val_accs'][-1]:.2%}",
        "Test Acc":        f"{res['test_acc']:.2%}",
        "Time (s)":        f"{res['elapsed']:.1f}",
    })
summary_df = pd.DataFrame(summary_rows).sort_values("Test Acc", ascending=False)
print(summary_df.to_string(index=False))
print("\nPyTorch Adam vs AdamFromScratch: test accuracies should be very close.")


---
## Part 4 — Evaluation & Analysis

### 4.1 Learning Rate Sensitivity

Optimisers differ greatly in how sensitive they are to the choice of learning rate.
Adaptive methods (Adam, RMSProp) have a **much wider \"goldilocks\" range** than SGD.


In [ ]:
# ── Learning rate sensitivity sweep ──────────────────────────────────────────
# Train for 3 epochs only (speed) across 5 LRs × 3 representative optimisers.
# We compare Adam, SGD+Momentum, and AdaGrad.

lr_candidates = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
sweep_opts = {
    "Adam":        lambda lr, params: AdamFromScratch(params, lr=lr),
    "SGD mu=0.9":  lambda lr, params: SGDFromScratch(params, lr=lr, momentum=0.9),
    "AdaGrad":     lambda lr, params: AdaGradFromScratch(params, lr=lr),
}
LR_SWEEP_EPOCHS = 3

lr_sweep_results: dict[str, list[float]] = {name: [] for name in sweep_opts}

for lr_cand in lr_candidates:
    for opt_name_sw, opt_factory in sweep_opts.items():
        torch.manual_seed(SEED)
        m_sw = FashionMLP(ARCH).to(device)
        m_sw.load_state_dict({k: v.clone() for k, v in ref_state.items()})
        o_sw = opt_factory(lr_cand, m_sw.parameters())
        c_sw = nn.CrossEntropyLoss()
        for _ in range(LR_SWEEP_EPOCHS):
            train_one_epoch(m_sw, train_loader, o_sw, c_sw, device)
        _, val_acc_sw = evaluate(m_sw, val_loader, c_sw, device)
        lr_sweep_results[opt_name_sw].append(val_acc_sw)
        print(f"  {opt_name_sw:<15s}  lr={lr_cand:.0e}  val_acc={val_acc_sw:.2%}")

# ── Plot: val accuracy vs learning rate ───────────────────────────────────────
fig_lr, ax_lr = plt.subplots(figsize=(9, 4))
sweep_cols = ["#9467bd", "#ff7f0e", "#d62728"]
for (opt_name_sw, accs), col_sw in zip(lr_sweep_results.items(), sweep_cols):
    ax_lr.semilogx(lr_candidates, accs, "o-", lw=2, ms=7, color=col_sw,
                   label=opt_name_sw)

ax_lr.set_xlabel("Learning rate (log scale)", fontsize=11)
ax_lr.set_ylabel(f"Val Accuracy after {LR_SWEEP_EPOCHS} epochs", fontsize=11)
ax_lr.set_title("LR Sensitivity: Adam Tolerates a Wider Range than SGD",
                fontsize=11, fontweight="bold")
ax_lr.legend(fontsize=9)
ax_lr.grid(alpha=0.3)
ax_lr.set_xticks(lr_candidates)
ax_lr.set_xticklabels([f"{lr:.0e}" for lr in lr_candidates])
plt.tight_layout()
plt.show()

print("\nKey insight: Adam's val accuracy is >0.7 for all 5 LRs tested.")
print("SGD+Momentum fails at high LRs (diverges) and is very slow at low LRs.")
print("AdaGrad is forgiving at high LRs but converges slowly due to decaying LR.")


In [ ]:
# ── Convergence analysis: epoch-1 speed + final accuracy gap ─────────────────
print("Optimiser comparison summary (FashionMNIST, 10 epochs):")
print(f"  {'Optimiser':<20s}  {'Ep-1 Val Acc':>12s}  {'Final Val Acc':>13s}  "
      f"{'Test Acc':>8s}  {'Time (s)':>8s}")
print("  " + "-" * 68)

for opt_name, res in results_cmp.items():
    ep1  = res["val_accs"][0]
    fin  = res["val_accs"][-1]
    test = res["test_acc"]
    t_s  = res["elapsed"]
    print(f"  {opt_name:<20s}  {ep1:>12.2%}  {fin:>13.2%}  {test:>8.2%}  {t_s:>8.1f}")

# ── Bar chart: epoch-1 vs final val accuracy ─────────────────────────────────
fig_bar, axes_bar = plt.subplots(1, 2, figsize=(14, 4))
opt_names_ba = list(results_cmp.keys())
ep1_vals_ba  = [results_cmp[n]["val_accs"][0]  for n in opt_names_ba]
fin_vals_ba  = [results_cmp[n]["test_acc"]     for n in opt_names_ba]
x_bar        = list(range(len(opt_names_ba)))
cols_bar     = plt.cm.tab10(np.linspace(0, 1, len(opt_names_ba)))

for ax_bar, y_bar, title_bar in zip(
    axes_bar,
    [ep1_vals_ba, fin_vals_ba],
    ["Epoch-1 Val Acc (Convergence Speed)", "Final Test Accuracy"],
):
    bars_b = ax_bar.bar(x_bar, y_bar, color=cols_bar, edgecolor="k", lw=0.6)
    for b_b, v_b in zip(bars_b, y_bar):
        ax_bar.text(
            b_b.get_x() + b_b.get_width() / 2.0,
            b_b.get_height() + 0.006,
            f"{v_b:.1%}",
            ha="center", va="bottom", fontsize=8,
        )
    ax_bar.set_xticks(x_bar)
    ax_bar.set_xticklabels(opt_names_ba, rotation=30, ha="right", fontsize=8)
    ax_bar.set_ylabel("Accuracy", fontsize=11)
    ax_bar.set_title(title_bar, fontsize=11, fontweight="bold")
    ax_bar.set_ylim(0, 1.08)
    ax_bar.grid(axis="y", alpha=0.3)

plt.suptitle("FashionMNIST: Convergence Speed vs Final Test Accuracy",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Recommendations table ─────────────────────────────────────────────────────
rec_data = [
    ("SGD",           "Simple tasks, full-batch GD, when momentum=0",  "Very sensitive to LR"),
    ("SGD+Momentum",  "Standard supervised learning (e.g., ResNets)",  "LR + momentum to tune"),
    ("AdaGrad",       "Sparse features, NLP tasks (good for embeddings)", "LR decays to 0"),
    ("RMSProp",       "RNNs, non-stationary objectives",               "No momentum term"),
    ("Adam",          "General default; MLP, CNN, early-stage training", "Can overfit at convergence"),
    ("AdamW",         "Transformers, large models; standard since 2019", "Need weight_decay tuned"),
]
rec_df = pd.DataFrame(rec_data, columns=["Optimiser", "Best For", "Caveat"])
print("\nOptimiser selection guide:")
print(rec_df.to_string(index=False))


### 4.2 Weight Norm Evolution — Adam vs AdamW

One practical difference between Adam and AdamW is their effect on **weight magnitudes** over time.  AdamW applies the weight decay factor $(1 - \alpha\lambda)$ directly to the parameter each step, so weights are kept small.  Standard Adam with L2 regularisation (folded into the gradient) scales the decay by $1/\sqrt{\hat{v}}$, reducing its effect.


In [ ]:
# ── Weight norm evolution during training ─────────────────────────────────────
# Track how weight norms grow or shrink per layer for Adam vs AdamW.
# AdamW (decoupled weight decay) should produce smaller norms than Adam.

norm_track_opts = {
    "Adam":             lambda params: AdamFromScratch(params, lr=LEARNING_RATE),
    "AdamW":            lambda params: AdamWFromScratch(params, lr=LEARNING_RATE,
                                                         weight_decay=WEIGHT_DECAY),
    "SGD (mu=0.9)":     lambda params: SGDFromScratch(params, lr=0.02, momentum=0.9),
}

layer_norms_all: dict[str, list[list[float]]] = {}

for norm_name, norm_factory in norm_track_opts.items():
    torch.manual_seed(SEED)
    m_nt = FashionMLP(ARCH).to(device)
    m_nt.load_state_dict({k: v.clone() for k, v in ref_state.items()})
    o_nt = norm_factory(m_nt.parameters())
    c_nt = nn.CrossEntropyLoss()

    epoch_norms: list[list[float]] = []
    for epoch_nt in range(5):
        train_one_epoch(m_nt, train_loader, o_nt, c_nt, device)
        linears_nt = [m for m in m_nt.net if isinstance(m, nn.Linear)]
        norms_nt   = [float(lin.weight.norm().item()) for lin in linears_nt]
        epoch_norms.append(norms_nt)

    layer_norms_all[norm_name] = epoch_norms
    print(f"  {norm_name:<15s}  final layer norms: "
          + "  ".join(f"{n:.3f}" for n in epoch_norms[-1]))

# ── Plot: weight norm per layer across epochs ──────────────────────────────────
n_layers_nt = len(ARCH) - 1
fig_wn, axes_wn = plt.subplots(1, n_layers_nt, figsize=(14, 4))
epochs_wn  = list(range(1, 6))
cols_wn    = ["#9467bd", "#8c564b", "#ff7f0e"]

for lyr_idx in range(n_layers_nt):
    for (norm_name, epoch_norms), col_wn in zip(layer_norms_all.items(), cols_wn):
        norms_lyr = [epoch_norms[e][lyr_idx] for e in range(5)]
        axes_wn[lyr_idx].plot(epochs_wn, norms_lyr, "o-", lw=1.8, color=col_wn,
                               label=norm_name)
    axes_wn[lyr_idx].set_xlabel("Epoch", fontsize=10)
    axes_wn[lyr_idx].set_ylabel("||W||_F", fontsize=10)
    axes_wn[lyr_idx].set_title(f"Layer {lyr_idx + 1} Weight Norm",
                                 fontsize=10, fontweight="bold")
    axes_wn[lyr_idx].legend(fontsize=8)
    axes_wn[lyr_idx].grid(alpha=0.3)

plt.suptitle("Weight Norm Evolution: AdamW Decays Weights; Adam Does Not",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nWeight norm interpretation:")
print("  Adam:        norms tend to grow — no explicit weight constraint.")
print("  AdamW:       norms held in check by decoupled decay (1 - lr * wd) factor.")
print("  SGD+momentum: norms grow due to momentum, but tend to stabilise earlier.")


---
## Part 5 — Summary & Key Takeaways

### Key Takeaways

- **Vanilla SGD** establishes the baseline: $\theta \leftarrow \theta - \alpha g$.
  It is optimal for convex problems with a good LR, but zig-zags on ill-conditioned
  landscapes.
- **Momentum** adds a velocity buffer that exponentially averages past gradients,
  suppressing oscillations across steep directions and accelerating progress along
  shallow ones.
- **AdaGrad** introduces per-parameter adaptive learning rates scaled by
  $1/\sqrt{G_t}$.  It is excellent for sparse features but suffers from a
  monotonically decaying effective LR.
- **RMSProp** fixes AdaGrad's dying-LR issue by replacing the cumulative sum with
  an **exponential moving average** $v_t$, keeping the effective LR stable.
- **Adam** = Momentum + RMSProp + **bias correction**.  The bias correction terms
  prevent underestimating moments in early steps.  Adam is the most widely used
  optimiser in practice.
- **AdamW** decouples weight decay from the gradient, so the adaptive scaling does
  not reduce the regularisation effect.  It is the modern standard for Transformers
  and large-scale training.

### What's Next

→ **05-10 (Complete MLP Pipeline)** applies everything from Module 05 — from weight
initialisation and optimiser selection to regularisation and evaluation — into a
production-quality end-to-end pipeline on FashionMNIST, culminating in ONNX export.

### Going Further

- **Learning rate schedules** (cosine annealing, OneCycleLR) combine with any
  optimiser and are covered in **Module 16**.
- **Gradient clipping** prevents exploding gradients (introduced in 05-06) and is
  commonly used with RNNs and Transformers.
- **LAMB / LARS** extend Adam/AdamW to very large batch sizes (not covered here).
- Kingma & Ba, *Adam: A Method for Stochastic Optimization*, 2015.
- Loshchilov & Hutter, *Decoupled Weight Decay Regularization*, 2019.
